# Разведочный анализ данных (EDA) — участники дорожного движения

Ноутбук к разделу отчёта **«Набор данных»**: анализ подмножества COCO,
отфильтрованного и ремапнутого в 5 пользовательских классов (пешеход,
велосипедист, мотоциклист, легковой автомобиль, грузовой транспорт).
Рассматриваются распределение классов, размеры объектов и аугментации.

Результаты используются в разделах «Набор данных» и «Обсуждение» отчёта.


## 1. Импорты, конфигурация и маппинг классов

In [ ]:
import sys, json
from pathlib import Path
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.utils import load_config

ROOT = Path("..").resolve()                    # корень проекта (пути в конфиге относительны ему)
cfg = load_config("../configs/default.yaml")
classes = cfg["classes"]
names = classes["names"]                       # имена 5 классов
mapping = classes["coco_mapping"]              # {индекс класса: [id категорий COCO]}
# обратное отображение: COCO category_id -> индекс нашего класса (0..4)
coco2cls = {int(cid): int(idx) for idx, ids in mapping.items() for cid in ids}
print("Классы проекта:", names)
print("COCO id -> класс:", coco2cls)

## 2. Загрузка аннотаций и фильтрация по 5 классам

Из аннотаций COCO (формат: `images`, `annotations` c bbox `[x, y, w, h]` и
`category_id`, `categories`) оставляем только объекты нужных категорий. Для
скорости используется val2017.

In [ ]:
ann_file = ROOT / cfg["data"]["val_ann"]
if ann_file.exists():
    coco = json.loads(ann_file.read_text(encoding="utf-8"))
    anns = [a for a in coco["annotations"] if a["category_id"] in coco2cls]
    print("Изображений в наборе:", len(coco["images"]))
    print("Аннотаций всего:     ", len(coco["annotations"]))
    print("Аннотаций 5 классов: ", len(anns))
else:
    print("Файл аннотаций не найден — положите COCO в data/raw/coco (см. README).")
    coco, anns = None, []

## 3. Распределение по пользовательским классам

In [ ]:
if anns:
    labels = [names[coco2cls[a["category_id"]]] for a in anns]
    counts = pd.Series(labels).value_counts().reindex(names).fillna(0).astype(int)
    fig, ax = plt.subplots(figsize=(9, 5))
    counts.plot.bar(ax=ax, color="#2c7fb8")
    ax.set_ylabel("Число объектов")
    ax.set_title("Распределение объектов по 5 классам (COCO val2017)")
    plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()
    print("Дисбаланс классов: max/min =", round(counts.max() / max(counts.min(), 1), 1))
    display(counts)

## 4. Распределение размеров объектов

По соглашению COCO объекты делятся на small (площадь < 32²), medium (32²–96²)
и large (> 96²). Доля мелких объектов важна: именно на них модели различаются
сильнее всего (см. раздел «Обсуждение» отчёта — мотоциклисты и велосипедисты вдали).

In [ ]:
if anns:
    areas = np.array([a["area"] for a in anns if a.get("area", 0) > 0])
    small = (areas < 32**2).mean()
    medium = ((areas >= 32**2) & (areas < 96**2)).mean()
    large = (areas >= 96**2).mean()
    print(f"small:  {small:.1%}\nmedium: {medium:.1%}\nlarge:  {large:.1%}")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(np.sqrt(areas), bins=60, color="#7fcdbb")
    ax.set_xlabel("√площадь объекта, px"); ax.set_ylabel("Частота")
    ax.set_title("Распределение размеров объектов (5 классов)")
    plt.tight_layout(); plt.show()

## 5. Визуализация примера с рамками (через DetectionDataset)

In [ ]:
try:
    import cv2
    import matplotlib.patches as patches
    from src.dataset.dataset import DetectionDataset

    ds = DetectionDataset(str(ROOT / cfg["data"]["val_images"]),
                          str(ROOT / cfg["data"]["val_ann"]),
                          transforms=None, subset=50,
                          coco_mapping=mapping, class_names=names)
    print("Изображений с нужными классами (первые 50):", len(ds))

    # берём первое существующее на диске изображение (часть файлов может быть не докачана)
    img_id = next(i for i in ds.ids
                  if (ds.images_dir / ds.images[i]["file_name"]).exists())
    img = cv2.cvtColor(cv2.imread(str(ds.images_dir / ds.images[img_id]["file_name"])),
                       cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(9, 6)); ax.imshow(img); ax.axis("off")
    for a in ds.anns[img_id]:
        x, y, w, h = a["bbox"]; cls = ds.cat2label[a["category_id"]]
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color="lime", lw=2))
        ax.text(x, y - 3, names[cls], color="black", fontsize=9,
                bbox=dict(facecolor="lime", alpha=0.7, pad=1, edgecolor="none"))
    plt.title("Пример изображения с рамками (5 классов)"); plt.show()
except Exception as e:
    print("Пропущено (нет данных/torch/opencv):", e)

## 6. Проверка аугментаций

Аугментации (flip, crop, color jitter) применяются только к train-выборке и
синхронно трансформируют рамки. Пайплайн задаётся в `build_transforms`.

In [ ]:
from src.dataset.dataset import build_transforms
tf = build_transforms(cfg, train=True)
print("Train-трансформации:", type(tf).__name__)
print("Настройки аугментаций:", cfg["preprocess"]["augmentation"])
print("Размер входа:", cfg["preprocess"]["img_size"])

## 7. Выводы EDA

- Набор сведён к **5 прикладным классам** участников дорожного движения; распределение
  сильно несбалансировано (~35×): доминирует класс «пешеход», а «велосипедист» и
  «мотоциклист» — самые редкие, что усложняет их детектирование.
- Значительная доля **мелких объектов** (порядка 40 %) объясняет, почему модели сильнее
  всего расходятся на удалённых участниках движения (велосипедисты, мотоциклисты вдали).
- Для обучения применяются нормализация по статистикам ImageNet и аугментации.

Эти наблюдения используются в разделах отчёта «Набор данных» и «Обсуждение».